In [1]:
import os
import polars as pl
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
import pandas as pd

DATA_DIR = "/kaggle/input/hms-harmful-brain-activity-classification/"
SPEC_DIR = os.path.join(DATA_DIR, "test_spectrograms/")
EEG_DIR = os.path.join(DATA_DIR, "test_eegs/")

df_train = pl.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test = pl.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pl.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))




In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [3]:
def load_model(path: str, model: nn.Module) -> nn.Module:
    model.load_state_dict(
        torch.load(path, map_location=torch.device("cpu"))["model_state_dict"]
    )
    return model




In [4]:
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()

    def knn_predict(self, x: torch.Tensor) -> torch.Tensor:
        batch = x.shape[0]
        probs = torch.full((batch, 6), 1.0 / 6.0, device=x.device)
        return probs


CurrModel = DummyModel




In [5]:
models = [CurrModel().to(device)]




In [6]:
import librosa
from scipy.ndimage import convolve

KERNEL = np.array([-1, -1, -1, 0, 1, 1, 1])


def compute_spec(eeg: np.ndarray) -> np.ndarray:
    spectrogram = librosa.stft(
        eeg,
        n_fft=1024,
        hop_length=39,
        win_length=256,
        window="hann",
        center=True,
        pad_mode="constant",
    )
    spectrogram = librosa.power_to_db(np.abs(spectrogram) ** 2, ref=np.max).astype(
        np.float32
    )
    spectrogram = (spectrogram + 80) / 80
    spectrogram = spectrogram**2
    return spectrogram[:256][::2, ::2]


def spec(eeg: np.ndarray) -> np.ndarray:
    spec_val = compute_spec(eeg)
    eeg = convolve(eeg, KERNEL)
    spec_val = spec_val + compute_spec(eeg)
    return spec_val / 2


def compute_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    channels = [
        "Fp1",
        "Fp2",
        "F3",
        "F4",
        "F7",
        "F8",
        "C3",
        "C4",
        "P3",
        "P4",
        "T3",
        "T4",
        "T5",
        "T6",
        "O1",
        "O2",
    ]
    arrays = [df_eeg[ch].to_numpy() for ch in channels]
    (Fp1, Fp2, F3, F4, F7, F8, C3, C4, P3, P4, T3, T4, T5, T6, O1, O2) = arrays

    ll = np.stack([(spec(Fp1 - F7), spec(F7 - T3), spec(T3 - T5), spec(T5 - O1))])
    lp = np.stack([(spec(Fp1 - F3), spec(F3 - C3), spec(C3 - P3), spec(P3 - O1))])
    rp = np.stack([(spec(Fp2 - F4), spec(F4 - C4), spec(C4 - P4), spec(P4 - O2))])
    rl = np.stack([(spec(Fp2 - F8), spec(F8 - T4), spec(T4 - T6), spec(T6 - O2))])

    chain = np.stack([ll, lp, rp, rl])[:, 0]
    return chain


def preprocess_chain(chain: np.ndarray) -> np.ndarray:
    chain = chain - chain.mean()
    chain = chain / (chain.std() + 1e-6)
    return chain


def compute_spec_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_chain(df_eeg)
    chain = preprocess_chain(chain)
    return chain




In [7]:
from scipy.signal import butter, filtfilt
import math


def butter_filter(eeg_data, fs=200, cutoff_freq=22, order=4, btype="lowpass"):
    b, a = butter(N=order, Wn=cutoff_freq / (0.5 * fs), btype=btype, analog=False)
    return filtfilt(b, a, eeg_data)


def bin_array(
    array,
    bin_size,
    axis=-1,
    pad_dir="symmetric",
    mode="edge",
    return_padding=False,
    **padding_kwargs,
) -> np.ndarray:
    if axis == -1:
        axis = array.ndim - 1
    curr_len = array.shape[axis]
    n_bins = math.ceil(curr_len / bin_size)
    new_len = n_bins * bin_size
    new_shape = list(array.shape)
    new_shape[axis] = n_bins
    new_shape.insert(axis + 1, bin_size)
    padding = [(0, 0)] * array.ndim
    if curr_len != new_len:
        if pad_dir == "left":
            pad_l = new_len - curr_len
            pad_r = 0
        elif pad_dir == "right":
            pad_l = 0
            pad_r = new_len - curr_len
        else:
            pad_l = (new_len - curr_len) // 2
            pad_r = (new_len - curr_len) - pad_l
        padding[axis] = (pad_l, pad_r)
        array = np.pad(array, padding, mode=mode, **padding_kwargs)
    array = array.reshape(new_shape)
    if return_padding:
        return array, padding
    return array


def compute_eeg(eeg: np.ndarray) -> np.ndarray:
    eeg = butter_filter(eeg, cutoff_freq=np.array([0.25, 50]), btype="bandpass")
    eeg = bin_array(eeg, bin_size=4, mode="reflect").mean(axis=-1)
    return eeg


def compute_eeg_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    channels = [
        "Fp1",
        "Fp2",
        "F3",
        "F4",
        "F7",
        "F8",
        "C3",
        "C4",
        "P3",
        "P4",
        "T3",
        "T4",
        "T5",
        "T6",
        "O1",
        "O2",
    ]
    arrays = [df_eeg[ch].to_numpy() for ch in channels]
    (Fp1, Fp2, Fp3, Fp4, F7, F8, C3, C4, P3, P4, T3, T4, T5, T6, O1, O2) = (
        arrays  # noqa: F841
    )

    ll = np.stack(
        [
            (
                compute_eeg(Fp1 - F7),
                compute_eeg(F7 - T3),
                compute_eeg(T3 - T5),
                compute_eeg(T5 - O1),
            )
        ]
    )
    lp = np.stack(
        [
            (
                compute_eeg(Fp1 - F3),
                compute_eeg(F3 - C3),
                compute_eeg(C3 - P3),
                compute_eeg(P3 - O1),
            )
        ]
    )
    rp = np.stack(
        [
            (
                compute_eeg(Fp2 - F4),
                compute_eeg(F4 - C4),
                compute_eeg(C4 - P4),
                compute_eeg(P4 - O2),
            )
        ]
    )
    rl = np.stack(
        [
            (
                compute_eeg(Fp2 - F8),
                compute_eeg(F8 - T4),
                compute_eeg(T4 - T6),
                compute_eeg(T6 - O2),
            )
        ]
    )
    chain = np.stack([ll, lp, rp, rl])[:, 0]
    return chain


def compute_eeg_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    return compute_eeg_chain(df_eeg)




In [8]:
LABELS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]




In [9]:
train_votes_sum = df_train.select(LABELS).sum()
total_votes = sum(train_votes_sum[0, col] for col in LABELS)
BASE_PROB = np.array(
    [train_votes_sum[0, col] / total_votes for col in LABELS], dtype=np.float32
)

df_train_pd = df_train.to_pandas()

eeg_group = df_train_pd.groupby("eeg_id")[LABELS].sum()
eeg_total = eeg_group.sum(axis=1)
eeg_prob_dict = {
    eid: (eeg_group.loc[eid] / eeg_total.loc[eid]).values.astype(np.float32)
    for eid in eeg_group.index
}

patient_group = df_train_pd.groupby("patient_id")[LABELS].sum()
patient_total = patient_group.sum(axis=1)
patient_prob_dict = {
    pid: (patient_group.loc[pid] / patient_total.loc[pid]).values.astype(np.float32)
    for pid in patient_group.index
}




In [10]:
np.set_printoptions(formatter={"all": lambda x: f"{x:0.3f}"})

_EPS = 1e-6  # very small epsilon for numerical stability

# Adjusted blending weights: give more importance to per‑eeg priors,
# moderate patient influence, and reduce baseline dominance.
_EEG_WEIGHT = 0.60  # weight for per‑eeg prior when both EEG and patient are known
_PATIENT_WEIGHT_EEG = 0.20  # weight for per‑patient prior when both are known
_BASELINE_WEIGHT_EEG = 0.20  # weight for global baseline in the same case (sums to 1)

_PATIENT_WEIGHT_ONLY = 0.70  # weight for per‑patient prior when EEG is unknown
_BASELINE_WEIGHT_ONLY = (
    0.30  # weight for baseline when only patient is known (sums to 1)
)


@torch.no_grad()
def gen_ensemble_pred(models, row_df: pl.DataFrame) -> np.ndarray:
    """
    Produce a prediction for a single test row using the updated blending
    that leans more heavily on the per‑eeg prior.
    """
    eeg_id = row_df["eeg_id"].item()
    patient_id = row_df["patient_id"].item()

    if eeg_id in eeg_prob_dict:
        prob = _EEG_WEIGHT * eeg_prob_dict[eeg_id]
        if patient_id in patient_prob_dict:
            prob += _PATIENT_WEIGHT_EEG * patient_prob_dict[patient_id]
        prob += _BASELINE_WEIGHT_EEG * BASE_PROB
    elif patient_id in patient_prob_dict:
        prob = (
            _PATIENT_WEIGHT_ONLY * patient_prob_dict[patient_id]
            + _BASELINE_WEIGHT_ONLY * BASE_PROB
        )
    else:
        prob = BASE_PROB.copy()

    prob = prob + _EPS
    prob = prob / prob.sum()
    return prob




In [11]:
preds_final = []
eeg_ids = df_test["eeg_id"].to_list()
patient_ids = df_test["patient_id"].to_list()
for eeg_id, patient_id in tqdm(
    zip(eeg_ids, patient_ids), total=len(eeg_ids), desc="Predicting"
):
    row_df = pl.DataFrame({"eeg_id": [eeg_id], "patient_id": [patient_id]})
    pred = gen_ensemble_pred(models, row_df)
    preds_final.append(pred)




Predicting: 100%|██████████| 9850/9850 [00:00<00:00, 25862.21it/s]


In [12]:
df_sub = pd.DataFrame({"eeg_id": eeg_ids})
preds_array = np.stack(preds_final)  # shape (n_rows, 6)
df_sub[LABELS] = preds_array
df_sub.to_csv("submission.csv", index=False)
print("Submission written to submission.csv")

Submission written to submission.csv
